# 02 Pipeline Template

Clone this notebook and replace the example source, transformation and target. Run in order: **Environment → Setup → Read → Transform → Write → Finalize**. Parameter tables below show the available options; runnable cells use defaults where practical.

## Tested with FabricOps

This notebook is part of the FabricOps release templates. Release snapshots preserve their corresponding notebook and package versions.

| FabricOps release | Tested by | Date tested |
| --- | --- | --- |
| v0.5.0 | Voyce | 16 Nov 2026 |


# 0. Environment

`setup_notebook()` loads `00_env_config` for the current workspace. Use `config_workspace_id="<workspace-id>"` only when configuration lives in another workspace.

In [ ]:
from fabricops_kit import finalize_pipeline, orchestrate_read, orchestrate_write, profile_table, setup_notebook, setup_pipeline

context = setup_notebook()

# 1. Setup Pipeline

`setup_pipeline()` sets execution and profiling behavior for the entire notebook. Profiling is independent of Data Contract Guardrails.

| Parameter | Default | Options / when to change |
|---|---|---|
| `pipeline_mode` | `"Ungoverned"` | `"Dry Run"` — no publication.<br>`"Ungoverned"` — publish without activated contracts.<br>`"Governed"` — activated contracts and trusted deployment required; currently blocked pending validation. |
| `profiling_enabled` | `True` | `False` disables standard profiling; does not disable Guardrails. |
| `profiling_interval_days` | `1` | Positive integer, e.g. `7` to reuse completed profiles up to seven days old. |
| `profiling_max_rows` | `None` | Full table or a positive row limit, e.g. `10000` for bounded **sample-scope** statistics. |
| `grain_discovery_enabled` | `False` | `True` opts into candidate Grain discovery; independent of Grain Guardrails. |
| `frequency_enabled` | `False` | `True` enables value-frequency profiling. |
| `source_drift_enabled` | `False` | `True` enables source-drift observations. |
| `source_drift_interval_days` | `1` | Positive integer, e.g. `7` for a seven-day source-drift interval. |
| `source_drift_block` | `True` | `False` warns rather than blocks on detected drift when enabled. |

**Mode behavior:** Dry Run previews without publishing or persisting; Ungoverned publishes without activated contracts; Governed requires activated contracts and a verified deployment. Source participation and reporting are automatic.

In [ ]:
PIPELINE_MODE = "Ungoverned"

pipeline = setup_pipeline(pipeline_mode=PIPELINE_MODE)

# 2. Read

Use the numbered Read code blocks below. **READ 1** runs by default; edit and uncomment **READ 2** or **READ 3** to add sources. Choose table coordinates or a Lakehouse file path.

| Parameter | Default | Options / when to change |
|---|---|---|
| **Source store** | | |
| `store` | **Required** | Configured source store alias (e.g. `"Bronze"`). |
| **Read from a table** | | |
| `schema`, `table_name` | `None` | E.g. `schema="demo"`, `table_name="products"`; omit schema for schemaless stores. |
| **Read from Lakehouse Files (alternative to table)** | | |
| `relative_path` | `None` | E.g. `"Files/import/products.csv"` or `"Files/import/*.csv"`. |
| `file_format` | `None` | E.g. `"csv"`, `"json"`, or `"parquet"`; omit to use inference when supported. |
| `read_options` | `None` | Optional Spark reader settings, e.g. `{"header": "true", "delimiter": ","}` for a CSV with column headers. |
| **Read behavior** | | |
| `read_mode` | `"full"` | `"full"` (all rows) or `"incremental"` (changes since checkpoint); files support only `"full"`. |
| `query` | `None` | E.g. `query="SELECT * FROM demo.products"` for a **full table** read (Spark SQL / T-SQL). Not for incremental or file reads. |
| `read_parameters` | `None` | Incremental-only dictionary; e.g. `{"watermark_column": "updated_at"}` where supported by the configured incremental strategy. |
| `target_store`, `target_schema`, `target_table_name` | `None` | Incremental checkpoint target, e.g. `target_store="Silver"`, `target_schema="demo"`, `target_table_name="products"`. Not needed for full reads. |
| **Diagnostics** | | |
| `verbose` | `False` | `True` for additional operational diagnostics. |
| `dq_diagnostics` | `"sample"` | `"none"` (metrics only), `"sample"` (sample failed rows), or `"full"` (all failed rows). |

In [ ]:
# READ 1 — Full table read
sources = {}

sources["read_1"] = orchestrate_read(
    store="Bronze",
    schema="demo",
    table_name="source_table",
    query="SELECT * FROM demo.source_table",
)

display(sources["read_1"]["dataframe"].limit(10))

In [ ]:
# READ 2 — Optional additional table read (edit and uncomment)
# sources["read_2"] = orchestrate_read(
#     store="Bronze",
#     schema="demo",
#     table_name="another_source_table",
# )
# display(sources["read_2"]["dataframe"].limit(10))

In [ ]:
# READ 3 — Optional Lakehouse Files read (edit and uncomment)
# sources["read_3"] = orchestrate_read(
#     store="Bronze",
#     relative_path="Files/demo/source.csv",
# )
# display(sources["read_3"]["dataframe"].limit(10))

# 3. Transform

Replace the example selection with your PySpark transformations. Reads and Writes are profiled according to pipeline settings; optional standalone `profile_table(dataframe=...)` does not persist governed profiles.

In [ ]:
transformed_df = sources["read_1"]["dataframe"].select("*")

# Optional development inspection.
# display(transformed_df)

# Optional: profile the transformed DataFrame before writing (no metadata persistence).
# transformed_profile = profile_table(dataframe=transformed_df)


# 4. Write

Use the numbered Write code blocks below. **WRITE 1** runs by default; edit and uncomment **WRITE 2** or **WRITE 3** to add targets. Choose table coordinates or a Lakehouse file path.

| Parameter | Default | Options / when to change |
|---|---|---|
| **Write input and destination store** | | |
| `dataframe` | **Required** | Prepared PySpark DataFrame, e.g. `transformed_df`. |
| `store` | **Required** | Configured target store alias (e.g. `"Silver"`). |
| **Write to a table** | | |
| `schema`, `table_name` | `None` | E.g. `schema="demo"`, `table_name="products"`; omit schema for schemaless stores. |
| **Write to Lakehouse Files (alternative to table)** | | |
| `relative_path` | `None` | E.g. `"Files/exports/products.csv"` (file) or `"Files/exports/products"` (folder). |
| `file_format` | `None` | E.g. `"csv"`, `"json"`, or `"parquet"` when writing Files. |
| `write_options` | `None` | Optional Spark writer settings, e.g. `{"header": "true", "delimiter": ","}` when exporting CSV. |
| `single_file` | `False` | `True` for one physical file (e.g. CSV export); `False` for a Spark dataset folder. |
| `max_single_file_rows` | `1000000` | E.g. `500000` to cap single-file output at 500,000 rows. |
| `max_single_file_bytes` | `134217728` | E.g. `67108864` for a 64 MiB single-file safety limit. |
| **Write behavior** | | |
| `write_mode` | **Required** | Tables: `"overwrite"`, `"append_all"`, `"append_new_only"`, `"scd1"`, `"scd2"`.<br>Files: `"errorifexists"`, `"overwrite"`, `"append"`, `"ignore"`. |
| `write_parameters` | `None` | Mode-specific dictionary, e.g. `{"key_columns": ["product_id"]}` for an SCD strategy; other keys depend on the mode. |
| `repartition_by` | `None` | Positive partition count, e.g. `8` to write using eight Spark partitions. |
| `existing_mapping` | `None` | Caller-managed Sensitive Data token mapping; normally leave unset for managed mappings. |
| **Diagnostics** | | |
| `verbose` | `False` | `True` for additional operational diagnostics. |
| `dq_diagnostics` | `"sample"` | `"none"` (metrics only), `"sample"` (sample failed rows), or `"full"` (all failed rows). |

In [ ]:
# WRITE 1 — Overwrite a table
writes = {}

writes["write_1"] = orchestrate_write(
    transformed_df,
    store="Silver",
    schema="demo",
    table_name="target_table",
    write_mode="overwrite",
)

display(writes["write_1"]["dataframe"].limit(10))

In [ ]:
# WRITE 2 — Optional additional table write (edit and uncomment)
# writes["write_2"] = orchestrate_write(
#     transformed_df,
#     store="Silver",
#     schema="demo",
#     table_name="another_target_table",
#     write_mode="overwrite",
# )
# display(writes["write_2"]["dataframe"].limit(10))

In [ ]:
# WRITE 3 — Optional Lakehouse Files write (edit and uncomment)
# writes["write_3"] = orchestrate_write(
#     transformed_df,
#     store="Silver",
#     relative_path="Files/demo/output",
#     file_format="parquet",
#     write_mode="overwrite",
# )
# display(writes["write_3"]["dataframe"].limit(10))

# 5. Finalize

Always call `finalize_pipeline()` after Reads and Writes. Dry Run keeps results in-session; publishing modes record the outcome and release publication claims.

In [ ]:
pipeline_result = finalize_pipeline(pipeline)

## Optional — Approve and deploy a Governed snapshot

1. **Engineering:** save the development notebook and run it in `"Dry Run"` through Finalize to save a submission.
2. **Governance:** open `widget_activate_pipeline()`, refresh submissions, review the saved snapshot and approve with a review note.
3. **Engineering:** refresh approvals under Finalize, review and deploy the approved notebook, then schedule the Governed notebook.
4. **Recovery:** choose a retained implementation or rollback request; Governance approves and Engineering applies the approval.

Governed execution remains blocked until trusted deployment verification is integrated. For detailed steps see the [governed pipeline guide](https://voycepeh.github.io/FabricOps-Starter-Kit/guided-demo/06-run-governed-pipeline/).